# Evaluation audit — provenance for the fit-for-purpose evaluation

Self-contained recomputation of every number in the paper's **evaluation** subsection
(Results, `subsec:eval`), directly from the simulation run outputs. **No imports from
`scripts/` or `cag`** — every helper is defined inline, so the notebook can be read and
audited end to end.

**How to run:** select the `.venv` kernel and *Run All*. Results are written to
`paper/tables/audit_evaluation.csv` (claim → paper value → recomputed → source runs →
status) and `paper/tables/audit_evaluation_log.txt` (full console log). Read those two
small files instead of scrolling the notebook.

**Sections (research title — internal tag):**
1. Construct validity: does the model read the persona it is given? — *Tier P*
2. Sensitivity: does opinion respond to a reach advantage? — *Tier 1 (reach)*
3. Robustness to the opinion scale, using ceiling-free rulers — *Tier 2*
4. Replication across social graphs and a second language model — *Tier 3*

Outcome metric: the within-person difference in the final **package index** (−3…+3),
paired per agent within seed and pooled over seeds. Sign convention throughout:
pro-climate-dominant condition − climate-sceptic-dominant condition.


In [1]:
# ---------------------------------------------------------------------------
# Setup & inline helpers (self-contained; no imports from scripts/ or cag)
# ---------------------------------------------------------------------------
from pathlib import Path
import glob, json
import numpy as np
import pandas as pd
from scipy import stats

# Locate the repo root by walking up until we find data/output.
REPO = Path.cwd()
while not (REPO / "data" / "output").exists() and REPO != REPO.parent:
    REPO = REPO.parent
EXP = REPO / "data" / "output" / "experiments"
TABLES = REPO / "paper" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
print("REPO           :", REPO)
print("experiments dir:", EXP, "(exists:", EXP.exists(), ")")
print("tables out dir :", TABLES)

POLICY_NAMES = {1: "Renewable energy", 2: "Ban fossil fuels", 3: "Ban petrol cars",
                4: "Green housing", 5: "Carbon tax", 6: "Climate compensation"}

# Audit log: everything we print is also captured to a text file.
LOG = []
def out(msg=""):
    print(msg)
    LOG.append(str(msg))

# Provenance rows: claim / paper value / recomputed value / source runs / status.
PROV = []
def check(claim, paper, computed, source, tol=0.02, kind="close"):
    if computed is None:
        ok = False
    elif kind == "ge":
        ok = float(computed) >= float(paper)
    else:
        ok = abs(float(computed) - float(paper)) <= tol
    status = "PASS" if ok else "CHECK"
    PROV.append({"claim": claim, "paper": paper,
                 "computed": None if computed is None else round(float(computed), 4),
                 "source_runs": source, "status": status})
    c = "n/a" if computed is None else f"{round(float(computed),4):+.4f}"
    out(f"  [{status}] {claim}")
    out(f"         paper={paper}   computed={c}   <- {source}")
    return ok

def info(claim, paper, computed, source):
    """Record a value for reference without pass/fail (canonical impl elsewhere)."""
    c = None if computed is None else round(float(computed), 4)
    PROV.append({"claim": claim, "paper": paper, "computed": c,
                 "source_runs": source, "status": "INFO"})
    out(f"  [INFO] {claim}: paper={paper}  computed={c}  <- {source}")

def latest_subdir(run_dir):
    subs = sorted([p for p in Path(run_dir).iterdir() if p.is_dir()])
    return subs[-1] if subs else Path(run_dir)

def find_run(base, token, seed):
    """Single run dir matching run_*_<token>_s<seed>, else None."""
    hits = sorted(glob.glob(str(Path(base) / f"run_*_{token}_s{seed}")))
    return hits[0] if hits else None

def find_by_id(base, id_glob):
    hits = sorted(glob.glob(str(Path(base) / id_glob)))
    return hits[0] if hits else None

def load_endpoints(run_dir):
    """Per-agent final-day package index + ground truth + exposure bucket."""
    d = latest_subdir(run_dir)
    pkg = pd.read_csv(d / "package_index_trajectories.csv")
    gt = pd.read_csv(d / "package_ground_truth.csv").set_index("agent_id")["ground_truth"]
    attr = pd.read_csv(d / "agent_attributes.csv").set_index("agent_id")["political_exposure"]
    dN = sorted(pkg["day"].unique())[-1]
    end = pkg[pkg.day == dN].set_index("agent_id")["package_index"]
    df = pd.DataFrame({"end": end, "gt": gt, "bucket": attr}).dropna()
    return df.reset_index()

def mean_ci(d):
    d = np.asarray(d, float)
    m = d.mean()
    h = d.std(ddof=1) / np.sqrt(len(d)) * stats.t.ppf(0.975, len(d) - 1)
    return m, m - h, m + h

def pooled_diff(base, tok_pro, tok_scep, seeds=(42, 43, 44), bucket=None):
    """Within-person endpoint diff (pro - scep), paired within seed, pooled over seeds."""
    chunks, per_seed, srcs = [], {}, []
    for s in seeds:
        rp, rs = find_run(base, tok_pro, s), find_run(base, tok_scep, s)
        if rp is None or rs is None:
            per_seed[s] = None
            continue
        srcs += [Path(rp).name, Path(rs).name]
        a, b = load_endpoints(rp), load_endpoints(rs)
        if bucket:
            a, b = a[a.bucket == bucket], b[b.bucket == bucket]
        m = a[["agent_id", "end", "gt"]].merge(b[["agent_id", "end"]],
                                               on="agent_id", suffixes=("_p", "_s"))
        d = (m["end_p"] - m["end_s"]).to_numpy()
        chunks.append(pd.DataFrame({"d": d, "gt": m["gt"].to_numpy()}))
        per_seed[s] = float(np.mean(d))
    if not chunks:
        return None
    D = pd.concat(chunks, ignore_index=True)
    m, lo, hi = mean_ci(D["d"])
    _, p = stats.ttest_1samp(D["d"], 0.0)
    return {"n": len(D), "mean": float(m), "lo": float(lo), "hi": float(hi), "p": float(p),
            "per_seed": {k: (None if v is None else round(v, 3)) for k, v in per_seed.items()},
            "src": sorted(set(srcs))}

out("helpers ready")


REPO           : /Users/ajaykumar/src/GitHub/Climate-Action-GABM
experiments dir: /Users/ajaykumar/src/GitHub/Climate-Action-GABM/data/output/experiments (exists: True )
tables out dir : /Users/ajaykumar/src/GitHub/Climate-Action-GABM/paper/tables
helpers ready


In [3]:
# ===========================================================================
# 1. CONSTRUCT VALIDITY: does the model read the persona it is given?
#    (internal: Tier P)   Runs: run_6457850..6457858 (NB 36/37)
#    Day-0 survey, no anchor, 3 persona modes (real / shuffled / neutral) x 3 seeds.
# ===========================================================================
import re

def pid_int(x):
    """policy_id may be an int or a string like 'ClimatePolicyID(1)'; extract the int."""
    m = re.search(r"(\d+)", str(x))
    return int(m.group(1)) if m else None

out("")
out("=" * 74)
out("1. CONSTRUCT VALIDITY - does the model read the persona? (internal: Tier P)")
out("   Runs: run_6457850..6457858 (NB 36/37); Day-0 survey, 3 persona modes x 3 seeds")
out("=" * 74)

tierp_ids = [f"run_{i}" for i in range(6457850, 6457859)]
rows, real_runs = [], []
for rid in tierp_ids:
    d = latest_subdir(EXP / rid)
    cfg = json.loads((d / "config.json").read_text())
    arm = cfg.get("persona_mode")
    cal = pd.read_csv(d / "calibration.csv")
    cal = cal[cal.day == 0].copy()
    cal["arm"] = arm
    cal["seed"] = cfg.get("random_seed")
    rows.append(cal)
    if arm == "real":
        real_runs.append(rid)
C = pd.concat(rows, ignore_index=True)

# (a) per-policy mean rank fidelity by arm (from calibration.csv)
rho_by_arm = C.groupby("arm")["spearman_rho"].mean()
out("per-policy mean Spearman rho by arm:")
out(rho_by_arm.round(3).to_string())

# (b) PACKAGE-level day-0 rank fidelity for the real arm (this is the ~0.62 the paper cites)
pkg_rhos = []
for rid in real_runs:
    d = latest_subdir(EXP / rid)
    pkg = pd.read_csv(d / "package_index_trajectories.csv")
    d0 = sorted(pkg.day.unique())[0]
    p0 = pkg[pkg.day == d0].set_index("agent_id")["package_index"]
    gt = pd.read_csv(d / "package_ground_truth.csv").set_index("agent_id")["ground_truth"]
    mm = pd.concat({"p": p0, "gt": gt}, axis=1).dropna()
    pkg_rhos.append(stats.spearmanr(mm["p"], mm["gt"]).correlation)
pkg_rho = float(np.mean(pkg_rhos)) if pkg_rhos else None

check("Tier P: real-persona rank fidelity (package index) [paper cites ~0.62]",
      0.62, pkg_rho, ",".join(real_runs), tol=0.06)
check("Tier P: real-persona rank fidelity (per-policy mean)",
      0.42, float(rho_by_arm.get("real", np.nan)), "run_6457850-6457858", tol=0.08)
if "neutral" in rho_by_arm.index:
    check("Tier P: no-persona rho ~ 0 (opinions collapse)",
          0.02, float(rho_by_arm["neutral"]), "run_6457850-6457858", tol=0.10)
if "shuffled" in rho_by_arm.index:
    check("Tier P: stranger-persona rho vs OWN opinion ~ 0",
          -0.10, float(rho_by_arm["shuffled"]), "run_6457850-6457858", tol=0.15)

pkg_bias = float(C[C.arm == "real"].groupby("seed")["mean_signed_bias"].mean().mean())
check("Tier P: residual pro-climate bias, real arm (package, points)",
      0.636, pkg_bias, "run_6457850-6457858", tol=0.15)

out("per-policy residual signed bias (real arm, points):")
bias_pol = C[C.arm == "real"].groupby("policy_id")["mean_signed_bias"].mean()
for pid, v in bias_pol.items():
    name = POLICY_NAMES.get(pid_int(pid), str(pid))
    out(f"     {name:22s} {v:+.3f}")

out("NOTE: the derangement value in the framing notes (rho +0.61 vs the assigned")
out("      STRANGER's opinion) needs the shuffle mapping and is computed in NB 36;")
out("      calibration.csv here carries rho-vs-OWN only (~0 above), which is what")
out("      the paper's 'collapses to near zero' sentence relies on.")



1. CONSTRUCT VALIDITY - does the model read the persona? (internal: Tier P)
   Runs: run_6457850..6457858 (NB 36/37); Day-0 survey, 3 persona modes x 3 seeds
per-policy mean Spearman rho by arm:
arm
neutral     0.017
real        0.418
shuffled   -0.099
  [PASS] Tier P: real-persona rank fidelity (package index) [paper cites ~0.62]
         paper=0.62   computed=+0.6212   <- run_6457850,run_6457851,run_6457852
  [PASS] Tier P: real-persona rank fidelity (per-policy mean)
         paper=0.42   computed=+0.4176   <- run_6457850-6457858
  [PASS] Tier P: no-persona rho ~ 0 (opinions collapse)
         paper=0.02   computed=+0.0171   <- run_6457850-6457858
  [PASS] Tier P: stranger-persona rho vs OWN opinion ~ 0
         paper=-0.1   computed=-0.0986   <- run_6457850-6457858
  [PASS] Tier P: residual pro-climate bias, real arm (package, points)
         paper=0.636   computed=+0.6356   <- run_6457850-6457858
per-policy residual signed bias (real arm, points):
     Renewable energy       +0.

In [4]:
# ===========================================================================
# 2. SENSITIVITY: does opinion respond to a reach advantage?
#    (internal: Tier 1, reach)  Runs: run_*_tier1_{baseline,green_dom,reform_dom,neither}_s{42,43,44}
#    green_dom = pro-climate side reaches more; reform_dom = sceptic side reaches more.
# ===========================================================================
out("")
out("=" * 74)
out("2. SENSITIVITY - does opinion respond to a reach advantage? (internal: Tier 1)")
out("   Runs: run_*_tier1_{baseline,green_dom,reform_dom,neither}_s{42,43,44}")
out("=" * 74)

reach = pooled_diff(EXP, "tier1_green_dom", "tier1_reform_dom")
check("Tier 1: reach effect (pro-dominant - sceptic-dominant), whole population",
      0.423, None if reach is None else reach["mean"],
      "" if reach is None else ",".join(reach["src"]), tol=0.03)
if reach:
    out(f"         95% CI [{reach['lo']:+.3f}, {reach['hi']:+.3f}]  p={reach['p']:.1e}  "
        f"per-seed={reach['per_seed']}  n={reach['n']}")

both = pooled_diff(EXP, "tier1_green_dom", "tier1_reform_dom", bucket="both")
check("Tier 1: among those who hear both sides (both bucket)",
      0.498, None if both is None else both["mean"], "tier1 both-bucket", tol=0.04)

neither = pooled_diff(EXP, "tier1_green_dom", "tier1_reform_dom", bucket="neither")
check("Tier 1: spillover to the no-broadcast bucket (two-step flow via peers)",
      0.248, None if neither is None else neither["mean"], "tier1 neither-bucket", tol=0.06)

# Placebo: each side vs the silent 'neither' CONDITION (no broadcasts at all).
g_pl = pooled_diff(EXP, "tier1_green_dom", "tier1_neither")
r_pl = pooled_diff(EXP, "tier1_reform_dom", "tier1_neither")
check("Tier 1: placebo - pro-side pushes UP vs the silent world",
      0.324, None if g_pl is None else g_pl["mean"], "tier1 green-vs-neither", tol=0.06)
check("Tier 1: placebo - sceptic-side pushes DOWN vs the silent world",
      -0.098, None if r_pl is None else r_pl["mean"], "tier1 reform-vs-neither", tol=0.06)

out("Interpretation: the whole-population +0.42 is diluted by citizens who hear one")
out("side or none; +0.50 among those who hear both sides is the effect where messages")
out("land; +0.25 among those who hear NO broadcast is genuine peer-network spillover.")
out("(Per-policy breadth across all six policies is audited in NB 39.)")



2. SENSITIVITY - does opinion respond to a reach advantage? (internal: Tier 1)
   Runs: run_*_tier1_{baseline,green_dom,reform_dom,neither}_s{42,43,44}
  [PASS] Tier 1: reach effect (pro-dominant - sceptic-dominant), whole population
         paper=0.423   computed=+0.4228   <- run_6458327_tier1_green_dom_s42,run_6458328_tier1_green_dom_s43,run_6458329_tier1_green_dom_s44,run_6458330_tier1_reform_dom_s42,run_6458331_tier1_reform_dom_s43,run_6458332_tier1_reform_dom_s44
         95% CI [+0.337, +0.508]  p=1.1e-19  per-seed={42: 0.532, 43: 0.423, 44: 0.313}  n=300
  [PASS] Tier 1: among those who hear both sides (both bucket)
         paper=0.498   computed=+0.4981   <- tier1 both-bucket
  [PASS] Tier 1: spillover to the no-broadcast bucket (two-step flow via peers)
         paper=0.248   computed=+0.2481   <- tier1 neither-bucket
  [PASS] Tier 1: placebo - pro-side pushes UP vs the silent world
         paper=0.324   computed=+0.3244   <- tier1 green-vs-neither
  [PASS] Tier 1: placebo

In [8]:
# ===========================================================================
# 3. ROBUSTNESS TO THE OPINION SCALE: ceiling-free rulers (internal: Tier 2)
#    Same 12 Tier-1 runs, re-expressed on rulers that a ceiling cannot distort.
#    The RAW ruler must equal the Tier-1 reach effect exactly (internal check);
#    the transformed rulers are reported for reference (canonical = NB 40).
# ===========================================================================
out("")
out("=" * 74)
out("3. ROBUSTNESS TO THE OPINION SCALE - ceiling-free rulers (internal: Tier 2)")
out("=" * 74)

def gather_pairs(base, tok1, tok2, seeds=(42, 43, 44)):
    frames = []
    for s in seeds:
        r1, r2 = find_run(base, tok1, s), find_run(base, tok2, s)
        if r1 is None or r2 is None:
            continue
        a, b = load_endpoints(r1), load_endpoints(r2)
        m = a[["agent_id", "end", "gt"]].merge(b[["agent_id", "end"]],
                                               on="agent_id", suffixes=("_g", "_r"))
        frames.append(m)
    return pd.concat(frames, ignore_index=True) if frames else None

P = gather_pairs(EXP, "tier1_green_dom", "tier1_reform_dom")
if P is not None:
    raw = (P["end_g"] - P["end_r"])

    # headroom ruler: movement as a fraction of the room above the start (3 - gt).
    # Agents already at the ceiling have ~no headroom, so exclude them (that IS the
    # point of the ruler - it only scores agents that could have moved up).
    room = 3.0 - P["gt"]
    hmask = room > 0.25
    head = ((P["end_g"] - P["end_r"])[hmask] / room[hmask])

    # logit ruler: stretch moves near the boundary of the bounded scale.
    def _logit(x):
        p = ((x + 3.0) / 6.0).clip(1e-4, 1 - 1e-4)
        return np.log(p / (1 - p))
    lg = _logit(P["end_g"]) - _logit(P["end_r"])

    # rank ruler: pure ordinal percentile on a common pooled ruler.
    allv = np.concatenate([P["end_g"].to_numpy(), P["end_r"].to_numpy()])
    rall = pd.Series(allv).rank(pct=True).to_numpy()
    rnk = rall[:len(P)] - rall[len(P):]

    check("Tier 2: RAW ruler reproduces the Tier-1 reach effect",
          0.423, float(raw.mean()), "tier1 (re-analysis)", tol=0.03)
    info("Tier 2: headroom ruler (ceiling-free, saturated agents excluded)",
         0.197, float(head.mean()), "tier1 re-analysis (NB 40 canonical)")
    info("Tier 2: logit ruler (ceiling-free)", 0.695, float(lg.mean()),
         "tier1 re-analysis (NB 40 canonical)")
    info("Tier 2: rank ruler (pure ordinal)", 0.088, float(rnk.mean()),
         "tier1 re-analysis (NB 40 canonical)")
    signs_ok = all(v > 0 for v in [float(raw.mean()), float(head.mean()),
                                   float(lg.mean()), float(rnk.mean())])
    out(f"All four rulers positive: {signs_ok}  => the reach effect is not an artefact")
    out("of the bounded -3..+3 scale. Transformed-ruler magnitudes depend on the exact")
    out("transform convention; NB 40 is the canonical implementation for the SI table.")
else:
    out("Tier-1 runs not found; skipping Tier 2 rulers.")



3. ROBUSTNESS TO THE OPINION SCALE - ceiling-free rulers (internal: Tier 2)
  [PASS] Tier 2: RAW ruler reproduces the Tier-1 reach effect
         paper=0.423   computed=+0.4228   <- tier1 (re-analysis)
  [INFO] Tier 2: headroom ruler (ceiling-free, saturated agents excluded): paper=0.197  computed=0.1896  <- tier1 re-analysis (NB 40 canonical)
  [INFO] Tier 2: logit ruler (ceiling-free): paper=0.695  computed=0.8638  <- tier1 re-analysis (NB 40 canonical)
  [INFO] Tier 2: rank ruler (pure ordinal): paper=0.088  computed=0.0876  <- tier1 re-analysis (NB 40 canonical)
All four rulers positive: True  => the reach effect is not an artefact
of the bounded -3..+3 scale. Transformed-ruler magnitudes depend on the exact
transform convention; NB 40 is the canonical implementation for the SI table.


In [6]:
# ===========================================================================
# 4. REPLICATION across social graphs and a second model (internal: Tier 3)
#    Network arm: Qwen3-14B on SBM (= Tier-1 runs) / Barabasi-Albert / Watts-Strogatz,
#    pooled over 3 seeds. Model arm: Llama-3.1-8B on SBM, single seed 42 (direction only).
# ===========================================================================
out("")
out("=" * 74)
out("4. REPLICATION across social graphs and a second model (internal: Tier 3)")
out("=" * 74)

sbm = pooled_diff(EXP, "tier1_green_dom", "tier1_reform_dom")
ba = pooled_diff(EXP, "tier3_ba_greendom", "tier3_ba_reformdom")
ws = pooled_diff(EXP, "tier3_ws_greendom", "tier3_ws_reformdom")
check("Tier 3 network: stochastic-block-model reach effect",
      0.423, None if sbm is None else sbm["mean"], "tier1 SBM", tol=0.03)
check("Tier 3 network: Barabasi-Albert (scale-free/hubs) reach effect",
      0.337, None if ba is None else ba["mean"], "" if ba is None else ",".join(ba["src"]), tol=0.05)
check("Tier 3 network: Watts-Strogatz (small-world) reach effect",
      0.360, None if ws is None else ws["mean"], "" if ws is None else ",".join(ws["src"]), tol=0.05)

# Model arm: Llama-3.1-8B, seed 42 (explicit run IDs; naming differs from token scheme).
def diff_by_ids(pro_glob, scep_glob):
    rp, rs = find_by_id(EXP, pro_glob), find_by_id(EXP, scep_glob)
    if rp is None or rs is None:
        return None
    a, b = load_endpoints(rp), load_endpoints(rs)
    m = a[["agent_id", "end"]].merge(b[["agent_id", "end"]], on="agent_id", suffixes=("_p", "_s"))
    d = m["end_p"] - m["end_s"]
    mm, lo, hi = mean_ci(d)
    return {"mean": float(mm), "lo": float(lo), "hi": float(hi), "n": len(d),
            "src": [Path(rp).name, Path(rs).name]}

llama = diff_by_ids("run_6479341*", "run_6479390*")
check("Tier 3 model: Llama-3.1-8B reach effect (single seed, direction only)",
      1.477, None if llama is None else llama["mean"],
      "" if llama is None else ",".join(llama["src"]), tol=0.08)
out("The reach effect survives a different social graph AND a different language model")
out("(even one that is a poorer simulator), because the within-person difference")
out("cancels each model's common-mode level error.")



4. REPLICATION across social graphs and a second model (internal: Tier 3)
  [PASS] Tier 3 network: stochastic-block-model reach effect
         paper=0.423   computed=+0.4228   <- tier1 SBM
  [PASS] Tier 3 network: Barabasi-Albert (scale-free/hubs) reach effect
         paper=0.337   computed=+0.3367   <- run_6475088_tier3_ba_greendom_s42,run_6475089_tier3_ba_reformdom_s42,run_6480780_tier3_ba_greendom_s43,run_6480781_tier3_ba_reformdom_s43,run_6480782_tier3_ba_greendom_s44,run_6480783_tier3_ba_reformdom_s44
  [PASS] Tier 3 network: Watts-Strogatz (small-world) reach effect
         paper=0.36   computed=+0.3600   <- run_6475091_tier3_ws_greendom_s42,run_6475092_tier3_ws_reformdom_s42,run_6480298_tier3_ws_reformdom_s43,run_6480300_tier3_ws_greendom_s44,run_6480301_tier3_ws_reformdom_s44,run_6480784_tier3_ws_greendom_s43
  [PASS] Tier 3 model: Llama-3.1-8B reach effect (single seed, direction only)
         paper=1.477   computed=+1.4767   <- run_6479341_tier3_llama8b_greendom_s42,run_

In [7]:
# ===========================================================================
# Provenance table + export (read these two files instead of the notebook)
# ===========================================================================
prov = pd.DataFrame(PROV, columns=["claim", "paper", "computed", "source_runs", "status"])
csv_path = TABLES / "audit_evaluation.csv"
log_path = TABLES / "audit_evaluation_log.txt"
prov.to_csv(csv_path, index=False)

n_pass = int((prov.status == "PASS").sum())
n_check = int((prov.status == "CHECK").sum())
n_info = int((prov.status == "INFO").sum())
summary = (f"SUMMARY  PASS={n_pass}  CHECK={n_check}  INFO={n_info}   "
           f"(CHECK rows need a look)")
out("")
out("=" * 74)
out(summary)
out("=" * 74)
log_path.write_text("\n".join(LOG))
print("WROTE", csv_path)
print("WROTE", log_path)
prov



SUMMARY  PASS=20  CHECK=0  INFO=3   (CHECK rows need a look)
WROTE /Users/ajaykumar/src/GitHub/Climate-Action-GABM/paper/tables/audit_evaluation.csv
WROTE /Users/ajaykumar/src/GitHub/Climate-Action-GABM/paper/tables/audit_evaluation_log.txt


,claim,paper,computed,source_runs,status
0,Tier P: real-persona rank fidelity (package in...,0.620,0.6212,"run_6457850,run_6457851,run_6457852",PASS
1,Tier P: real-persona rank fidelity (per-policy...,0.420,0.4176,run_6457850-6457858,PASS
2,Tier P: no-persona rho ~ 0 (opinions collapse),0.020,0.0171,run_6457850-6457858,PASS
3,Tier P: stranger-persona rho vs OWN opinion ~ 0,-0.100,-0.0986,run_6457850-6457858,PASS
4,"Tier P: residual pro-climate bias, real arm (p...",0.636,0.6356,run_6457850-6457858,PASS
5,Tier P: real-persona rank fidelity (package in...,0.620,0.6212,"run_6457850,run_6457851,run_6457852",PASS
6,Tier P: real-persona rank fidelity (per-policy...,0.420,0.4176,run_6457850-6457858,PASS
7,Tier P: no-persona rho ~ 0 (opinions collapse),0.020,0.0171,run_6457850-6457858,PASS
8,Tier P: stranger-persona rho vs OWN opinion ~ 0,-0.100,-0.0986,run_6457850-6457858,PASS
9,"Tier P: residual pro-climate bias, real arm (p...",0.636,0.6356,run_6457850-6457858,PASS
